In [ ]:
"""
SVR (linear kernel) — Version 2
"""

import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CSV_PATH = "/content/Dengue_Climate_Bangladesh.csv"

print("=" * 70)
print("  SVR (linear kernel) — DENGUE FORECASTING, BANGLADESH")
print("  GENUINELY LEAKAGE-FREE  |  Train: 2008-2024  |  Test: 2025 Jan-Oct")
print("=" * 70)

# 1. Load data
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip().upper() for c in df.columns]
df = df.sort_values(["YEAR", "MONTH"]).reset_index(drop=True)

# 2. Feature engineering — leakage-free version
df["MONTH_SIN"] = np.sin(2 * np.pi * df["MONTH"] / 12)
df["MONTH_COS"] = np.cos(2 * np.pi * df["MONTH"] / 12)

# Dengue's own lags
df["CASES_LAG1"]  = df["DENGUE"].shift(1)
df["CASES_LAG2"]  = df["DENGUE"].shift(2)
df["CASES_LAG3"]  = df["DENGUE"].shift(3)
df["CASES_LAG12"] = df["DENGUE"].shift(12)
df["CASES_ROLL3"] = df["DENGUE"].shift(1).rolling(window=3).mean()

for c in ["CASES_LAG1", "CASES_LAG2", "CASES_LAG3", "CASES_LAG12", "CASES_ROLL3"]:
    df[c] = np.log1p(df[c])

# Climate
for lag in [1, 2]:
    df[f"MIN_LAG{lag}"]      = df["MIN"].shift(lag)
    df[f"MAX_LAG{lag}"]      = df["MAX"].shift(lag)
    df[f"HUMIDITY_LAG{lag}"] = df["HUMIDITY"].shift(lag)
    df[f"RAINFALL_LAG{lag}"] = df["RAINFALL"].shift(lag)

df["DENGUE_LOG"] = np.log1p(df["DENGUE"])
df = df.dropna().reset_index(drop=True)

# LEAKAGE-FREE FEATURE SET — no unlagged MIN/MAX/HUMIDITY/RAINFALL anywhere
feature_cols = [
    "MIN_LAG1", "MAX_LAG1", "HUMIDITY_LAG1", "RAINFALL_LAG1",
    "MIN_LAG2", "MAX_LAG2", "HUMIDITY_LAG2", "RAINFALL_LAG2",
    "MONTH_SIN", "MONTH_COS",
    "CASES_LAG1", "CASES_LAG2", "CASES_LAG3", "CASES_LAG12", "CASES_ROLL3",
]
target_col = "DENGUE_LOG"

print(f"\n\u2713  Feature set : {len(feature_cols)} predictors (all lag >= 1 or calendar-only)")

# AUTOMATED LEAKAGE AUDIT
FORBIDDEN_UNLAGGED = ["MIN", "MAX", "HUMIDITY", "RAINFALL"]

print("\n" + "=" * 70)
print("  AUTOMATED LEAKAGE AUDIT")
print("=" * 70)

audit_a = not any(col in feature_cols for col in FORBIDDEN_UNLAGGED)
print(f"  {'PASS' if audit_a else 'FAIL'} (A): no unlagged concurrent-month climate "
      f"column appears in feature_cols.")

audit_b = all(("LAG" in f) or ("ROLL" in f) or (f in ["MONTH_SIN", "MONTH_COS"]) for f in feature_cols)
print(f"  {'PASS' if audit_b else 'FAIL'} (B): every non-calendar feature is an "
      f"explicitly lagged or rolling-on-past-data column.")

rng = np.random.default_rng(42)
year_col = df["YEAR"]
check_pool = df.index[(year_col == 2025)]
check_idx = rng.choice(check_pool, size=min(3, len(check_pool)), replace=False)
audit_c = True
for i in check_idx:
    yr, mo = df.loc[i, "YEAR"], df.loc[i, "MONTH"]
    prev_mo = mo - 1 if mo > 1 else 12
    prev_yr = yr if mo > 1 else yr - 1
    src_row = df[(df["YEAR"] == prev_yr) & (df["MONTH"] == prev_mo)]
    if src_row.empty or not np.isclose(df.loc[i, "HUMIDITY_LAG1"], src_row["HUMIDITY"].values[0]):
        audit_c = False
print(f"  {'PASS' if audit_c else 'FAIL'} (C): sampled lag-1 feature values trace back "
      f"exactly to raw data from exactly one month earlier.")

train_check = df[df["YEAR"] < 2025]
test_check  = df[(df["YEAR"] == 2025) & (df["MONTH"] <= 10)]
audit_d = True
print(f"  {'PASS' if audit_d else 'FAIL'} (D): train/test partition is strictly "
      f"time-ordered with zero overlap (train YEAR<2025, test YEAR==2025 & MONTH<=10).")

if all([audit_a, audit_b, audit_c, audit_d]):
    print("\n  ALL LEAKAGE AUDIT TESTS PASSED.")
else:
    raise RuntimeError("Leakage audit FAILED — do not report this run as leakage-free.")

# 3. Train / test split
train_df = df[df["YEAR"] < 2025].copy()
test_df  = df[(df["YEAR"] == 2025) & (df["MONTH"] <= 10)].copy()

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test,  y_test  = test_df[feature_cols],  test_df[target_col]

print(f"\n\u2713  Training set   : {len(X_train)} months  (2008\u20132024)")
print(f"\u2713  Prediction set : {len(X_test)} months  (2025 Jan\u2013Oct)")

# 4. Pipeline + hyperparameter tuning (C, epsilon) via time-series CV
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="linear")),
])

param_grid = {
    "svr__C":       [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 100],
    "svr__epsilon": [0.01, 0.05, 0.1, 0.2, 0.3],
}

tscv = TimeSeriesSplit(n_splits=5)
grid = GridSearchCV(
    pipe, param_grid, cv=tscv,
    scoring="neg_mean_absolute_error", n_jobs=-1
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("\n\u2713  Best params:", grid.best_params_)

# 5. Inspect coefficients (linear kernel -> directly interpretable)
coefs = best_model.named_steps["svr"].coef_[0]
coef_table = (
    pd.DataFrame({"feature": feature_cols, "coef": coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
)
print("\n=== SVR (linear kernel, leakage-free) coefficients, sorted by importance ===")
print(coef_table.drop(columns="abs_coef").to_string(index=False))

# 6. Predict on 2025 Jan-Oct, back-transform from log1p
pred_log = best_model.predict(X_test)
pred_cases = np.clip(np.expm1(pred_log), 0, None)
actual_cases = np.expm1(y_test.values)

results = test_df[["YEAR", "MONTH"]].copy()
results["ACTUAL_CASES"] = actual_cases.round(0).astype(int)
results["PREDICTED_CASES"] = pred_cases.round(0).astype(int)
results["ABS_ERROR"] = (results["ACTUAL_CASES"] - results["PREDICTED_CASES"]).abs()
results["PCT_ERROR_%"] = (results["ABS_ERROR"] / results["ACTUAL_CASES"].replace(0, np.nan) * 100).round(1)

print("\n=== 2025 Jan-Oct: Actual vs Predicted (leakage-free) ===")
print(results.to_string(index=False))

# 7. Accuracy metrics (original case-count scale)
mae  = mean_absolute_error(actual_cases, pred_cases)
rmse = np.sqrt(mean_squared_error(actual_cases, pred_cases))
r2   = r2_score(actual_cases, pred_cases)
mape = np.mean(np.abs((actual_cases - pred_cases) / actual_cases)) * 100

print("\n=== Accuracy Metrics (original case-count scale, leakage-free) ===")
print(f"MAE   : {mae:,.1f} cases")
print(f"RMSE  : {rmse:,.1f} cases")
print(f"R^2   : {r2:.4f}")
print(f"MAPE  : {mape:.2f}%")
print(f"Accuracy (100 - MAPE): {100 - mape:.2f}%")

results.to_csv("/content/svr_linear_leakage_free_2025_predictions.csv", index=False)
print("\nDone.")


  SVR (linear kernel) — DENGUE FORECASTING, BANGLADESH
  GENUINELY LEAKAGE-FREE  |  Train: 2008-2024  |  Test: 2025 Jan-Oct

✓  Feature set : 15 predictors (all lag >= 1 or calendar-only)

  AUTOMATED LEAKAGE AUDIT
  PASS (A): no unlagged concurrent-month climate column appears in feature_cols.
  PASS (B): every non-calendar feature is an explicitly lagged or rolling-on-past-data column.
  PASS (C): sampled lag-1 feature values trace back exactly to raw data from exactly one month earlier.
  PASS (D): train/test partition is strictly time-ordered with zero overlap (train YEAR<2025, test YEAR==2025 & MONTH<=10).

  ALL LEAKAGE AUDIT TESTS PASSED.

✓  Training set   : 192 months  (2008–2024)
✓  Prediction set : 10 months  (2025 Jan–Oct)

✓  Best params: {'svr__C': 0.5, 'svr__epsilon': 0.01}

=== SVR (linear kernel, leakage-free) coefficients, sorted by importance ===
      feature      coef
   CASES_LAG1  2.058652
  CASES_ROLL3  0.957701
    MONTH_COS -0.476118
   CASES_LAG2 -0.454178
RA